# bot

> Discord passthrough to a headless coding agent CLI: subprocess runner,
> reply shaping, env config, and the thin discord.py glue.

In [ ]:
#| default_exp bot

## `run_agent`

Run one agent turn as a subprocess: stdin ← message, capture stdout and
stderr **separately** (claude emits warnings on stderr even on success).
Timeout: SIGKILL the whole process group, recover partial output via a
shielded `communicate()` (an unshielded `wait_for` drops the buffer).
Never raises — nonzero exit is data, not an exception, and is **never
retried** (a retry would replay side effects like PR creation).

In [ ]:
#| export
import asyncio, os, signal

async def run_agent(text:str,            # message for the agent, fed to stdin
                    cmd:list,            # agent command line, already shlex-split
                    cwd:str,             # working directory for the agent
                    timeout:float=600.0, # wall-clock seconds before SIGKILL
                    )->tuple:            # (stdout, stderr, returncode)
    "Run one agent turn; never raises on nonzero exit or timeout."
    proc = await asyncio.create_subprocess_exec(
        *cmd, cwd=cwd, start_new_session=True,
        stdin=asyncio.subprocess.PIPE,
        stdout=asyncio.subprocess.PIPE, stderr=asyncio.subprocess.PIPE)
    comm = asyncio.ensure_future(proc.communicate(text.encode()))
    try:
        out, err = await asyncio.wait_for(asyncio.shield(comm), timeout)
    except asyncio.TimeoutError:
        try: os.killpg(proc.pid, signal.SIGKILL)   # whole session: agent + children
        except ProcessLookupError: pass
        try:
            out, err = await asyncio.wait_for(asyncio.shield(comm), 1)
        except asyncio.TimeoutError:
            comm.cancel()
            return "… timed out (output withheld by a background child)\n", "", -9   # -9: killed by SIGKILL
        rc = proc.returncode if proc.returncode is not None else -9
        return out.decode(errors="replace") + "\n… timed out\n", err.decode(errors="replace"), rc
    return out.decode(errors="replace"), err.decode(errors="replace"), proc.returncode

In [ ]:
out, err, rc = await run_agent("hello", ["bash", "-c", "cat; echo warn >&2"], ".", 5)
assert out == "hello", out                    # stdin delivered, stdout captured
assert "warn" in err and "warn" not in out    # stderr kept separate
assert rc == 0

out, err, rc = await run_agent("", ["bash", "-c", "echo partial; echo boom >&2; exit 3"], ".", 5)
assert rc == 3                                # nonzero exit reported, not raised
assert "partial" in out and "boom" in err

out, err, rc = await run_agent("", ["bash", "-c", "echo start; sleep 5"], ".", 0.5)
assert "start" in out, out                    # partial output recovered
assert "timed out" in out and rc != 0

out, err, rc = await run_agent("", ["bash", "-c", "echo hi; sleep 60 &"], ".", 1)
assert "hi" in out and "timed out" in out     # killpg reaps the background child

## `compose` and `reply_text`

Success → stdout verbatim (it's markdown; Discord renders it, no fences).
Failure → stdout, then `(exit N)` and the fenced stderr — that's where the
reason lives. Inner ``` in stderr is neutralized with a zero-width space so
it can't close the fence. `reply_text` tail-truncates to Discord's 2000-char
cap: the primary length control is a CLAUDE.md instruction in the agent's
cwd (operational, see spec), so this is the safety net.

In [ ]:
#| export
_ZWSP = "​"   # breaks a literal ``` so stderr can't close the fence

def compose(out:str,    # agent stdout (markdown)
            err:str,    # agent stderr, captured separately
            rc:int,     # agent exit code
            )->str:     # reply body for Discord
    "stdout alone on success; append `(exit N)` + fenced stderr on failure."
    if not rc: return out
    body = err.strip().replace("```", f"`{_ZWSP}`{_ZWSP}`")
    return f"{out}\n(exit {rc})\n```\n{body}\n```"

def reply_text(text:str,        # reply body (markdown, not fenced)
               limit:int=2000,  # Discord message length cap
               )->str:          # len() <= limit, never empty
    "Tail-keeping truncation: the end of an answer matters most."
    body = text.strip()
    if not body: return "(no output)"
    if len(body) <= limit: return body
    marker = "… (truncated)\n"
    avail = limit - len(marker)
    if avail <= 0: return marker[:limit]   # degenerate cap: marker alone, clipped
    return marker + body[-avail:]

In [ ]:
assert compose("ok", "warn on stderr", 0) == "ok"     # success: stderr never leaks
c = compose("partial", "boom", 3)
assert c.startswith("partial") and "(exit 3)" in c and "boom" in c
c = compose("x", "evil ``` fence", 1)
assert c.count("```") == 2                            # only compose's own fence pair

assert reply_text("hi") == "hi"
assert reply_text("") == "(no output)"
assert reply_text("  \n") == "(no output)"
long = "\n".join(f"line {i}" for i in range(500))
out = reply_text(long)
assert len(out) <= 2000
assert out.startswith("… (truncated)\n") and out.endswith("line 499")  # tail survives
assert len(reply_text(long, limit=100)) <= 100
assert len(reply_text(long, limit=14)) <= 14    # limit == marker length: degenerate but bounded
assert len(reply_text(long, limit=5)) <= 5      # limit < marker length: still bounded

## `load_config`

Environment variables only. Missing/malformed required var → clean
`sys.exit` naming the variable (lands in the systemd journal).
`DISCOPIPE_CWD` is **required**: `--continue` resumes "the latest
conversation in this directory", so a `$HOME` default would silently
collide with the operator's interactive claude sessions.

In [ ]:
#| export
import os, sys

DEFAULT_CMD = "claude -p --continue --dangerously-skip-permissions"

def load_config()->dict:  # token, user_id, channel_id, cwd, cmd, timeout
    "Read discopipe config from the environment; exit with a message if incomplete."
    e = os.environ
    try:
        token, cwd = e["DISCORD_TOKEN"], e["DISCOPIPE_CWD"]
        user_raw, chan_raw = e["DISCOPIPE_USER_ID"], e["DISCOPIPE_CHANNEL_ID"]
    except KeyError as ex:
        sys.exit(f"discopipe: missing environment variable {ex.args[0]}")
    try: user_id = int(user_raw)
    except ValueError: sys.exit("discopipe: DISCOPIPE_USER_ID must be an integer")
    try: channel_id = int(chan_raw)
    except ValueError: sys.exit("discopipe: DISCOPIPE_CHANNEL_ID must be an integer")
    try: timeout = float(e.get("DISCOPIPE_TIMEOUT", "600"))
    except ValueError: sys.exit("discopipe: DISCOPIPE_TIMEOUT must be a number")
    return dict(token=token, user_id=user_id, channel_id=channel_id, cwd=cwd,
                cmd=e.get("DISCOPIPE_CMD", DEFAULT_CMD), timeout=timeout)

In [ ]:
import os
_env = dict(DISCORD_TOKEN="tok", DISCOPIPE_USER_ID="1", DISCOPIPE_CHANNEL_ID="2",
            DISCOPIPE_CWD="/srv/agent")
for k in ("DISCOPIPE_CMD","DISCOPIPE_TIMEOUT"): os.environ.pop(k, None)
os.environ.update(_env)
cfg = load_config()
assert cfg == dict(token="tok", user_id=1, channel_id=2,
                   cwd="/srv/agent", cmd=DEFAULT_CMD, timeout=600.0)

os.environ.update(DISCOPIPE_CMD="codex exec", DISCOPIPE_TIMEOUT="30")
cfg = load_config()
assert (cfg["cmd"], cfg["timeout"]) == ("codex exec", 30.0)
for k in ("DISCOPIPE_CMD","DISCOPIPE_TIMEOUT"): os.environ.pop(k)

del os.environ["DISCORD_TOKEN"]
try: load_config(); assert False, "expected SystemExit"
except SystemExit as e: assert "DISCORD_TOKEN" in str(e)

os.environ.update(_env); del os.environ["DISCOPIPE_CWD"]
try: load_config(); assert False, "expected SystemExit"
except SystemExit as e: assert "DISCOPIPE_CWD" in str(e)   # required: no $HOME default

os.environ.update(_env); os.environ["DISCOPIPE_USER_ID"] = "not-a-number"
try: load_config(); assert False, "expected SystemExit"
except SystemExit as e: assert "DISCOPIPE_USER_ID must be an integer" in str(e)

os.environ.update(_env); os.environ["DISCOPIPE_CHANNEL_ID"] = "xyz"
try: load_config(); assert False, "expected SystemExit"
except SystemExit as e: assert "DISCOPIPE_CHANNEL_ID must be an integer" in str(e)

os.environ.update(_env); os.environ["DISCOPIPE_TIMEOUT"] = "soon"
try: load_config(); assert False, "expected SystemExit"
except SystemExit as e: assert "DISCOPIPE_TIMEOUT must be a number" in str(e)

os.environ.pop("DISCOPIPE_TIMEOUT")
for k in _env: os.environ.pop(k, None)